# Introduction

This Notebook introduces Gemma 4:E2B (it).

Recently Gemma 4 was released. Their capabilities include OCR, speech-to-text, object detection. They also support text-only, multimodal function calling, as well as reasoning, code completion, and correction.

Here are the features of each model:

* Gemma 4 E2B — has a context window of 128K. Parameter size is 2.3B effective, 5.1B with embeddings.
* Gemma 4 E4B — has a context window of 128K. Parameter size is 4.5B effective, 8B with embeddings.
* Gemma 4 26B — mixture of experts with 4B activated/26 total parameters. Context window in 256K.
* Gemma 4 31B — it is a 31B dense model, with context window 256K.
All models have both base and it versions. We will use Ollama on a local machine to run Gemma 4 E2B.



# Upgrade transformers

In [1]:
!pip install -U transformers  -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 85.9 MB/s eta 0:00:00


# Import packages

In [2]:
from time import time
from IPython.display import Markdown
import requests
from PIL import Image
import kagglehub
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

# Initialize model

In [3]:
MODEL_PATH = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e2b-it")

processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map="auto"
)

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

# Test with text input

In [4]:
messages = [
    {"role": "system", "content": "You are a helpful assistant that specializes in answering shortly to any question."},
    {"role": "user", "content": "What is the distance from Earth to the Moon?"},
]

s_time = time()
text = processor.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True, 
    enable_thinking=True
)
inputs = processor(text=text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]


outputs = model.generate(**inputs, max_new_tokens=1024)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

processor.parse_response(response)
e_time = time()
total_time = round(e_time - s_time, 2)
print(f"Total time: {total_time}")

Total time: 103.99


Let's display the output more user friendly.

In [5]:
output = processor.parse_response(response)

# Format the output

In [6]:
def colorize_text(text):
    for word, color in zip(["Thinking", "Thinking Process", "Response", "Total time"], ["blue", "red", "green", "magenta"]):
        text = text.replace(f"{word}:", f"\n\n**<font color='{color}'>{word}:</font>**")
    return text
    
def display_response(output, total_time):
    display(Markdown(f"<font color='green'>**Thinking**</font>"))
    display(Markdown(colorize_text(output["thinking"])))
    display(Markdown(f"<font color='blue'>**Response**</font>"))
    display(Markdown(output["content"]))
    display(Markdown(colorize_text(f"Total time: {total_time} sec.")))

In [7]:
display_response(output, total_time)

<font color='green'>**Thinking**</font>



**<font color='red'>Thinking Process:</font>**

1.  **Analyze the Request:** The user is asking for the distance from Earth to the Moon.
2.  **Identify the Goal:** Provide a short, accurate answer.
3.  **Recall/Verify Information:** The average distance from the Earth to the Moon is approximately 384,400 kilometers or 238,900 miles.
4.  **Format the Response (Short Answer Style):** State the distance clearly.

*(Self-Correction/Refinement: Keep it concise as per the instruction "answer shortly.")*

5.  **Final Output Generation.**

<font color='blue'>**Response**</font>

The average distance from Earth to the Moon is about 384,400 kilometers (238,900 miles).



**<font color='magenta'>Total time:</font>** 103.99 sec.

# Test with image

In [8]:
image_url = "https://storage.googleapis.com/keras-cv/models/paligemma/cow_beach_1.png"
image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "What you can see in this image?"}
        ]
    }
]
s_time = time()

text = processor.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True, 
    enable_thinking=True
)
inputs = processor(
    text=text,
    images=image,
    return_tensors="pt"
).to(model.device)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=1024)

response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
output = processor.parse_response(response)

e_time = time()
total_time = round(e_time - s_time, 2)

print(output)
print(f"Total time: {total_time}")

{'role': 'assistant', 'thinking': 'Here\'s a thinking process to generate the image description:\n\n1.  **Analyze the Request:** The user has provided an image and asked, "What you can see in this image?"\n\n2.  **Examine the Image (Initial Scan & Subject Identification):**\n    *   **Main Subject:** A cow (or bovine animal).\n    *   **Appearance of the Cow:** It has reddish-brown/tawny coat, and a prominent white patch on its face/head. It looks healthy.\n    *   **Setting/Environment:** Outdoors. There is sand in the foreground. There is water (the ocean/sea) in the background. The sky is bright blue with some white clouds.\n    *   **Lighting/Mood:** Bright, sunny, tropical/summer feel.\n\n3.  **Detailed Analysis (Focusing on Composition and Elements):**\n    *   **Foreground:** The cow is standing on wet/damp sand near the water\'s edge. Shadows are visible on the sand.\n    *   **Midground:** The cow itself, positioned facing slightly towards the viewer.\n    *   **Background:**\

Let's display the output nicely formated.

In [9]:
display_response(output, total_time)

<font color='green'>**Thinking**</font>

Here's a thinking process to generate the image description:

1.  **Analyze the Request:** The user has provided an image and asked, "What you can see in this image?"

2.  **Examine the Image (Initial Scan & Subject Identification):**
    *   **Main Subject:** A cow (or bovine animal).
    *   **Appearance of the Cow:** It has reddish-brown/tawny coat, and a prominent white patch on its face/head. It looks healthy.
    *   **Setting/Environment:** Outdoors. There is sand in the foreground. There is water (the ocean/sea) in the background. The sky is bright blue with some white clouds.
    *   **Lighting/Mood:** Bright, sunny, tropical/summer feel.

3.  **Detailed Analysis (Focusing on Composition and Elements):**
    *   **Foreground:** The cow is standing on wet/damp sand near the water's edge. Shadows are visible on the sand.
    *   **Midground:** The cow itself, positioned facing slightly towards the viewer.
    *   **Background:**
        *   The ocean/sea is visible, appearing turquoise or light blue near the shore.
        *   A distant landmass (hills or islands) is visible on the horizon.
        *   The sky dominates the upper portion, clear blue with wispy clouds.

4.  **Synthesize the Description (Drafting the response):** Structure the description logically (Subject -> Setting -> Atmosphere).

    *   *(Self-Correction/Refinement during drafting):* Ensure the description is vivid and captures the essence of the scene. Mention the colors and the overall mood.

5.  **Final Polish (Review against the image):** (The resulting description should accurately reflect all key visual elements.)

    *   *Key elements confirmed:* Brown and white cow, beach, ocean, blue sky, sunny day.

6.  **Generate Final Output.** (This matches the detailed response provided below.)

<font color='blue'>**Response**</font>

Based on the image provided, here is what can be seen:

**Main Subject:**
*   A cow (or bovine animal) is the central focus of the image.
*   The cow has a reddish-brown coat and a distinct white patch on its face/forehead.
*   It is standing on the shore, looking towards the camera.

**Setting and Environment:**
*   **Foreground:** There is light-colored sand, which appears slightly wet or damp near the water's edge.
*   **Midground:** The cow is standing on the beach.
*   **Background:** There is a body of water (likely the ocean or sea) with turquoise and blue hues near the shore. In the distance, there is a strip of land or hills on the horizon.
*   **Sky:** The sky is bright blue with some scattered white clouds, indicating a sunny day.

**Overall Impression:**
The image captures a bright, sunny outdoor scene, likely at a tropical or coastal location, featuring a striking contrast between the warm tones of the cow and the cool blues of the sea and sky.



**<font color='magenta'>Total time:</font>** 639.71 sec.

# Preliminary conclusions


We tested Gemma 4:E2B on CPU for text, image and multi-language capabilities. Although slow compared with GPU (tens of second for text only compared with less than a second on GPU, and several minutes for an image interpretation), the response is good quality.

We will follow with a version using GPU to compare the performance from execution time perspective.
